In [ ]:
#| default_exp spec

In [ ]:
#| export
from __future__ import annotations

In [ ]:
#| export
import importlib.util, os, subprocess, sys

In [ ]:
#| export
from dataclasses import dataclass, field

In [ ]:
#| export
from pathlib import Path

In [ ]:
#| export
from fastcore.all import L, first

In [ ]:
#| export
from jupyter_client.kernelspec import KernelSpec

In [ ]:
#| export
from kunda.support import HOST_PY, clean_env, support_paths

In [ ]:
#| export
def run_file_src(path, src=None, argv=(), cwd=None):
    "Code that runs a file in the kernel's *own* namespace, with script semantics."
    src = open(path, encoding='utf-8').read() if src is None else src
    setup = ('import os as _kd_os, sys as _kd_sys\n'
        f'_kd_code = compile({src!r}, {str(path)!r}, "exec")\n'
        '_kd_argv, _kd_cwd = _kd_sys.argv, _kd_os.getcwd()\n'
        f'_kd_sys.argv = [{str(path)!r}, *{list(argv)!r}]\n')
    if cwd: setup += f'_kd_os.chdir({str(cwd)!r})\n'
    return setup + ('try: exec(_kd_code)\n'
        'finally:\n'
        '    _kd_sys.argv = _kd_argv\n'
        '    _kd_os.chdir(_kd_cwd)\n'
        '    del _kd_os, _kd_sys, _kd_code, _kd_argv, _kd_cwd\n')

In [ ]:
#| export
BOOTSTRAP = '''
def _kd_bootstrap():
	import sys, importlib.util
	# Borrowed only by a kernel of the host's own minor version. A frozen build ships bytecode-only
	# modules in its zip, and another version reads their magic number and refuses: an import that
	# fell through to them died as `bad magic number`, not as anything about the inspector.
	if tuple(sys.version_info[:2]) == tuple({version!r}):
		for _p in {support!r}:
			if _p and _p not in sys.path: sys.path.append(_p)
	# The project venv may contain an older dhrishti. Pin the inspector protocol to this
	# host installation while every other import stays project-local. Best-effort: a frozen
	# build imports dhrishti out of a zip, where there is no file to point at. A failed pin
	# must leave nothing in sys.modules, or the fallback import finds the broken module.
	try:
		if tuple(sys.version_info[:2]) != tuple({version!r}): raise ImportError('another Python')
		for _n in [n for n in sys.modules if n == 'dhrishti' or n.startswith('dhrishti.')]:
			del sys.modules[_n]
		_spec = importlib.util.spec_from_file_location('dhrishti', {dhrishti_init!r},
			submodule_search_locations=[{dhrishti_dir!r}])
		_pkg = importlib.util.module_from_spec(_spec)
		sys.modules['dhrishti'] = _pkg
		_spec.loader.exec_module(_pkg)
	except Exception:
		for _n in [n for n in sys.modules if n == 'dhrishti' or n.startswith('dhrishti.')]:
			del sys.modules[_n]
	import dhrishti.serving as ls
	r = ls.serve_in_kernel(name={name!r}, port={port!r}, agent={agent!r}, token={token!r},
	                       session_dir={sessions!r}, agent_session_dir={agent_sessions!r})
	if {ipymini!r}:
		try:
			from IPython import get_ipython
			ls.set_kernel_backend(getattr(get_ipython(), 'kernel', None))
		except Exception: ls.set_kernel_backend(None)
	return r
try: _kd_bootstrap()
finally: del _kd_bootstrap
'''

In [ ]:
#| export
def bootstrap_src(name=None, port=8000, agent='restricted', token=True, kernel='ipykernel',
                  sessions='_kunda_sessions', agent_sessions='_kunda_agent_sessions'):
    "The bootstrap cell source for a kernel that should host an inspector."
    support = support_paths()
    # Found, not imported: importing dhrishti for its `__file__` drags IPython, pandas and
    # numpy into the host process, which has no use for them. The kernel does the importing.
    spec = importlib.util.find_spec('dhrishti')
    if spec is None or not spec.origin: raise RuntimeError('dhrishti is not installed')
    dhrishti_init = str(Path(spec.origin).resolve())
    return BOOTSTRAP.format(name=name, port=port, agent=agent, token=token, support=support,
        sessions=sessions, agent_sessions=agent_sessions,
        version=HOST_PY, dhrishti_init=dhrishti_init,
        dhrishti_dir=str(Path(dhrishti_init).parent), ipymini=(kernel == 'ipymini'))

In [ ]:
#| export
def output_text(outs):
    "Flatten nbformat outputs to plain text: the terminal rendering, and the agent's view of a run."
    parts = L()
    for o in outs:
        t = o.get('output_type')
        if t == 'stream': parts.append(o.get('text', ''))
        elif t == 'error': parts.append('\n'.join(o.get('traceback') or [f"{o.get('ename')}: {o.get('evalue')}"]))
        elif t in ('execute_result', 'display_data'):
            d = o.get('data') or {}
            parts.append(d.get('text/markdown') or d.get('text/plain') or next((f'[{k}]' for k in d), ''))
    return ''.join(parts)

In [ ]:
#| export
@dataclass
class ExecOutcome:
    "Result of one execute_request: nbformat-shaped outputs plus the shell reply status."
    ok: bool = True
    execution_count: int | None = None
    outputs: list = field(default_factory=list)
    error: str | None = None
    @property
    def text(self): return output_text(self.outputs)

In [ ]:
#| export
def _runtime_python(python=None):
    "A selected interpreter, or py2app's bundled generic Python helper."
    if python: return str(python)
    if getattr(sys, 'frozen', False):
        helper = Path(sys.executable).with_name('python')
        if helper.exists(): return str(helper)
    return sys.executable

In [ ]:
#| export
class KernelStartError(RuntimeError):
    "A kernel that could not start, said in terms of the environment rather than the protocol."

In [ ]:
#| export
def missing_kernel_module(python=None, kernel='ipykernel'):
    """The kernel package `python` cannot import, or None.

    A venv without it exits before the first message, and jupyter_client can only report that the
    kernel died before replying to `kernel_info`. Asked here, the environment says which module it
    is short of. Only the failure path pays for this.
    """
    mod = 'ipymini' if kernel == 'ipymini' else 'ipykernel_launcher'
    exe = _runtime_python(python)
    if exe == sys.executable: return None if importlib.util.find_spec(mod) else mod
    src = f"import importlib.util, sys; sys.exit(0 if importlib.util.find_spec({mod!r}) else 1)"
    try: r = subprocess.run([exe, '-c', src], capture_output=True, timeout=20, env=clean_env())
    except (OSError, subprocess.SubprocessError): return None   # cannot tell, so do not say
    return None if r.returncode == 0 else mod

In [ ]:
#| export
def _frozen_pythonpath():
    "Module and extension paths a py2app helper process must inherit."
    if not getattr(sys, 'frozen', False): return None
    resources = Path(sys.executable).resolve().parent.parent/'Resources'
    version = f'python{sys.version_info.major}.{sys.version_info.minor}'
    bundled = [resources/'lib'/version, resources/'lib'/version/'lib-dynload',
        resources/'lib'/f'python{sys.version_info.major}{sys.version_info.minor}.zip']
    return os.pathsep.join(dict.fromkeys(str(p) for p in [*bundled, *sys.path] if p))

In [ ]:
#| export
def _kernel_env(python=None):
    "Environment for a kernel child, detached from py2app's interpreter redirect."
    env = clean_env()
    if python is None and (path := _frozen_pythonpath()): env['PYTHONPATH'] = path
    return env

In [ ]:
#| export
def _spec(python=None, name='python3', kernel='ipykernel', lang='python', known=None, install=''):
    """A kernelspec pinned to a specific interpreter, so the kernel runs in *this* venv.

    Python is the language an interpreter is pinned for. Any other language runs whatever
    kernelspec is installed for it, unmodified, because there is no venv of ours to point it at."""
    if lang and lang != 'python': return installed_spec(lang, known, install)
    runtime = _runtime_python(python)
    if kernel == 'ipymini': argv = [runtime, '-Xfrozen_modules=off', '-m', 'ipymini', '-f', '{connection_file}']
    else: argv = [runtime, '-m', 'ipykernel_launcher', '-f', '{connection_file}']
    metadata = {'supported_encryption': 'curve'} if kernel == 'ipykernel' else {}
    return KernelSpec(argv=argv, display_name=name, language='python', metadata=metadata)

In [ ]:
#| export
def installed_kernels():
    "Every Jupyter kernelspec on this machine, as `{name: spec}`. An unreadable store is none."
    from jupyter_client.kernelspec import KernelSpecManager
    try: return KernelSpecManager().get_all_specs()
    except Exception: return {}

In [ ]:
#| export
def kernelspec_for(lang, known=None):
    """The installed kernelspec name that runs `lang`, or None.

    `known` is a caller's own `{language: kernelspec}` mapping, which wins where it answers: a host
    with a language registry knows which of several installed kernels it means.
    """
    specs = installed_kernels()
    if (name := (known or {}).get(str(lang))) and name in specs: return name
    return first(n for n, s in specs.items()
                 if str(((s.get('spec') or {}).get('language') or '')).lower() == str(lang).lower())

In [ ]:
#| export
def installed_spec(lang, known=None, install=''):
    "The `KernelSpec` for `lang`, or a `KernelStartError` naming what would install one."
    from jupyter_client.kernelspec import KernelSpecManager
    if (name := kernelspec_for(lang, known)): return KernelSpecManager().get_kernel_spec(name)
    how = f' Install one with `{install}`.' if install else ''
    raise KernelStartError(f'no Jupyter kernel is installed for {lang}.{how}')